# Fine-tune MultiVerS on our reranker's evidence

Continues training the authors' released MultiVerS checkpoint on top-20 evidence from our
reranker (`scripts/finetune_multivers.py`). The training split's evidence comes from 3-fold
cross-validated rerankers and is already committed, so this notebook only fine-tunes and scores.

Why: the BiomedBERT classifier trained on this evidence beats the paper on accuracy but not on
macro-F1, because it cannot learn NOT ENOUGH INFO from 143 examples (`results/classifier_v2.md`).
MultiVerS already handles that class; this run adapts it to the better evidence.

**Before running:** Runtime → Change runtime type → **T4 GPU** (an A100 is about 3x faster if
you have Colab Pro). Then Runtime → Run all, and allow Google Drive access when asked. On a T4
expect roughly 1.5 to 2 hours. Results are saved to `MyDrive/CiteAudit_runs/mv_ft` as each seed
finishes; if Colab disconnects, Run all again and finished seeds are skipped.

The bar: the released checkpoint gives test macro-F1 **0.524** (paper 0.52) and accuracy 0.584.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
RES = '/content/drive/MyDrive/CiteAudit_runs/mv_ft'
import os; os.makedirs(RES, exist_ok=True)

## Setup

Clones this repository and downloads the authors' released checkpoint (5.25 GB, a few minutes)
from their public Google Drive folder. If the download fails with a quota error, open
[their folder](https://drive.google.com/drive/folders/11b6Z8iv2FXObWmLaqfYzgUQsaL4QgTT2),
right-click `bestModel.ckpt` → Make a copy, and set `CKPT` below to
`/content/drive/MyDrive/Copy of bestModel.ckpt`.

In [ ]:
%cd /content
!rm -rf CiteAudit && git clone -q https://github.com/SapnilPatel/CiteAudit.git
%cd /content/CiteAudit
!git log -1 --oneline
!pip -q install "transformers>=4.44" "tokenizers>=0.19" gdown

CKPT = '/content/bestModel.ckpt'
if not os.path.exists(CKPT):
    !gdown -q 1fnNiY_ECuOgDt561S6pmc8hzI4rnjn5Y -O {CKPT}
!ls -la {CKPT}

## Fine-tune

Each seed first scores the unchanged checkpoint on dev (epoch 0), then fine-tunes for 2 epochs.
The epoch with the best dev macro-F1 is kept, including epoch 0, so a run that never improves on
the released model reports the released model rather than a worse one. Start with one seed;
add 1 and 2 to `SEEDS` if the first one is promising.

In [ ]:
import subprocess
def run(cmd):
    noise = ('Warning', 'warn(', 'mean_resizing', 'vocab-expansion', 'padded to be a multiple')
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if not any(n in line for n in noise):
            print(line, end='')
    if p.wait():
        raise RuntimeError(f'failed ({p.returncode}): {cmd}')

SEEDS = (0,)
for seed in SEEDS:
    out = f'{RES}/C_mvft_s{seed}'
    if os.path.exists(f'{out}/paper_20_test.jsonl'):
        print(f'seed {seed}: already done'); continue
    run(f'python scripts/finetune_multivers.py --checkpoint {CKPT} '
        f'--train data_train/target/ours_20_train_cv --dev data_train/target/ours_20 '
        f'--predict test=data_train/target/ours_20_test --predict test=data_train/target/paper_20_test '
        f'--pred-dir {out} --epochs 2 --seed {seed} --grad-checkpointing --amp')

## Results

The fine-tuned model on test, with our reranker's evidence (the setting it was trained for) and
with the paper's evidence.

In [ ]:
import json
LABELS = ['ACCURATE', 'NOT_ACCURATE', 'NOT ENOUGH INFO']
def scores(path):
    rows = [json.loads(l) for l in open(path)]
    f1 = []
    for c in LABELS:
        tp = sum(r['gold'] == c and r['prediction'] == c for r in rows)
        fp = sum(r['gold'] != c and r['prediction'] == c for r in rows)
        fn = sum(r['gold'] == c and r['prediction'] != c for r in rows)
        f1.append(2 * tp / (2 * tp + fp + fn) if tp else 0.)
    return [sum(r['gold'] == r['prediction'] for r in rows) / len(rows), sum(f1) / 3] + f1

print(f'{"run":12s} {"test evidence":14s} {"acc":>6s} {"macro":>6s} {"ACC":>6s} {"N_ACC":>6s} {"NEI":>6s}  best epoch')
print(f'{"paper":12s} {"paper":14s} {0.59:6.3f} {0.52:6.3f} {0.69:6.3f} {0.43:6.3f} {0.42:6.3f}')
print(f'{"MultiVerS":12s} {"paper":14s} {0.584:6.3f} {0.524:6.3f} {0.692:6.3f} {0.426:6.3f} {0.455:6.3f}')
for seed in SEEDS:
    out = f'{RES}/C_mvft_s{seed}'
    if not os.path.exists(f'{out}/config.json'): continue
    cfg = json.load(open(f'{out}/config.json'))
    for test in ('ours_20_test', 'paper_20_test'):
        s = scores(f'{out}/{test}.jsonl')
        print(f'{"ft seed " + str(seed):12s} {test[:-8]:14s} ' + ' '.join(f'{x:6.3f}' for x in s)
              + f'  {cfg["best_epoch"]}  (dev by epoch: '
              + ', '.join(f'{l["dev_macro_f1"]:.3f}' for l in cfg['log']) + ')')

## Download

Zips `MyDrive/CiteAudit_runs/mv_ft` (predictions and training logs, no weights). Put the unzipped
`mv_ft` folder in the repository root and tell Claude.

In [ ]:
!cd /content/drive/MyDrive/CiteAudit_runs && zip -qr /content/mv_ft.zip mv_ft
from google.colab import files
files.download('/content/mv_ft.zip')